# AE Pretraining Ablation Runner

Runs Stage 1 and optional Stage 1B experiments only. This notebook compares AE reconstruction quality before spending time on Stage 3/4/5 downstream runs.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match the working Colab setup used by Notebook 2:
# code repo in /content/neurovlm_gnn, Drive used for data and run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

from atlas_free_cnn.training.train_autoencoder import train_from_config, train_stage1b_from_config
from atlas_free_cnn.pipeline_outputs import create_full_pipeline_run_dir, write_table


In [ ]:
TRAIN_JSONL = "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits/train.jsonl"
VAL_JSONL = "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits/val.jsonl"
TEST_JSONL = "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits/test.jsonl"
AE_CHECKPOINT_SELECTION = "best_top5_dice"
RUN_STAGE1B = True
ABLATION_OUTPUT_DIR = DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation" if "DRIVE_ROOT" in globals() else Path("runs_atlas_free_cnn_ae_ablation")
paths = create_full_pipeline_run_dir(ABLATION_OUTPUT_DIR, prefix="ae_ablation")
RUN_DIR = Path(paths["run_dir"])
print(RUN_DIR)

In [ ]:
recipes = [
    {
        "ae_variant": "mixed_baseline_raw_mse",
        "ae_training_recipe": "baseline_raw_mse",
        "source_sampling": "natural",
        "loss": {"type": "raw_mse", "lambda_foreground": 0.0, "lambda_topk": 0.0, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_val_loss",
    },
    {
        "ae_variant": "mixed_balanced_raw_mse",
        "ae_training_recipe": "mixed_balanced_raw_mse",
        "source_sampling": "balanced",
        "loss": {"type": "raw_mse", "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
    {
        "ae_variant": "mixed_balanced_hybrid_loss",
        "ae_training_recipe": "mixed_balanced_hybrid_loss",
        "source_sampling": "balanced",
        "loss": {"type": "hybrid_recon", "lambda_foreground": 0.10, "lambda_topk": 0.05, "topk_percent": 5, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
]

In [ ]:
results = []
for recipe in recipes:
    out_dir = RUN_DIR / "01_stage1_ae_pretraining" / recipe["ae_variant"]
    cfg = {
        "train_jsonl": TRAIN_JSONL,
        "val_jsonl": VAL_JSONL,
        "test_jsonl": TEST_JSONL,
        "output_dir": str(out_dir),
        "checkpoint_dir": str(out_dir / "checkpoints"),
        "data_mode": "mixed",
        "target_shape": [36, 45, 38],
        "model": {"latent_dim": 384, "base_channels": 64, "num_blocks": 4, "dropout": 0.1, "norm": "group", "pooling": "max"},
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "amp": True,
        "gradient_clipping": 1.0,
        "batch_size": 64,
        "epochs": 300,
        **recipe,
    }
    result = train_from_config(cfg)
    results.append({"ae_variant": recipe["ae_variant"], "checkpoint_dir": result["checkpoint_dir"], "best_checkpoint": result["best_checkpoint"]})
results

In [ ]:
if RUN_STAGE1B and results:
    seed_checkpoint = results[-1]["best_checkpoint"]
    for mode, domain in [("mixed_pretrain_to_pubmed", "pubmed"), ("mixed_pretrain_to_statmaps", "statmaps")]:
        out_dir = RUN_DIR / "02_stage1b_ae_finetuning" / domain
        cfg = {
            "stage1b_mode": mode,
            "mixed_pretrain_checkpoint": seed_checkpoint,
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "output_dir": str(out_dir),
            "checkpoint_dir": str(out_dir / "checkpoints"),
            "ae_variant": "mixed_to_pubmed" if domain == "pubmed" else "mixed_to_statmaps",
            "lr": 1e-4,
            "freeze_mode": "none",
            "epochs": 100,
            "loss": {"type": "raw_mse", "prediction_activation": "none"},
        }
        ft_result = train_stage1b_from_config(cfg)
        results.append({"ae_variant": cfg["ae_variant"], "checkpoint_dir": ft_result["checkpoint_dir"], "best_checkpoint": ft_result["best_checkpoint"]})

In [ ]:
summary_rows = []
for row in results:
    metrics_path = Path(row["checkpoint_dir"]).parent / "metrics" / "reconstruction_summary_by_source.csv"
    metrics = {}
    if metrics_path.exists():
        with metrics_path.open(newline="") as f:
            for m in csv.DictReader(f):
                if m.get("split") in {"val", "test"} and m.get("source_detail") == "ALL_DETAILS" and m.get("source") in {"pubmed", "neurovault", "nilearn"}:
                    src = m["source"]
                    metrics[f"{src}_spatial_corr"] = m.get("spatial_corr", "")
                    metrics[f"{src}_top5_dice"] = m.get("top5_dice", "")
    values = []
    for key, value in metrics.items():
        try:
            values.append(float(value))
        except Exception:
            pass
    ranking_score = sum(values) / len(values) if values else ""
    summary_rows.append({**row, **metrics, "ranking_score": ranking_score, "metrics_path": str(metrics_path), "recommendation_note": "Prioritize spatial_corr/top5_dice by source; do not rank by MSE alone."})
summary_rows = sorted(summary_rows, key=lambda r: float(r["ranking_score"]) if r["ranking_score"] != "" else -999, reverse=True)
write_table(RUN_DIR / "07_final_comparison" / "best_checkpoints_to_inspect.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "final_summary_table.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "ae_ablation_leaderboard.csv", summary_rows)
print("AE ablation summary written to", RUN_DIR / "07_final_comparison")
summary_rows
